In [ ]:
import numpy as np
from manim import *

import icm_anim as anim

In [ ]:
class PressureWave(Scene):
    """Sound as a traveling pressure wave, read at one fixed point. A piston
    drives a longitudinal wave through a band of air particles. A red probe
    marks the microphone; the pressure it reads draws itself as x(t) below."""

    def construct(self):
        LAM = 4.0          # wavelength in scene units
        FREQ = 0.5         # Hz: one crest passes every 2 s
        DISP = 0.28        # longitudinal displacement amplitude
        Y_LO, Y_HI = 0.7, 2.9
        PISTON_X = -6.75
        PROBE_X = 2.5
        T_TRACE = 5.0      # when the oscilloscope starts drawing
        T_END = 13.0

        clock = ValueTracker(0.0)

        def disp(x0, t):
            return DISP * np.sin(TAU * (FREQ * t - x0 / LAM))

        def pressure(t):
            # normalized pressure at the probe: p is proportional to -dxi/dx
            return np.cos(TAU * (FREQ * t - PROBE_X / LAM))

        # the band of air
        rng = np.random.default_rng(15322)
        n_part = 380
        xs = rng.uniform(-6.5, 7.2, n_part)
        ys = rng.uniform(Y_LO, Y_HI, n_part)
        particles = VGroup(*[
            Dot([x, y, 0], radius=0.035, color=anim.IRON, fill_opacity=0.9)
            for x, y in zip(xs, ys)
        ])

        def move_particles(group):
            t = clock.get_value()
            for d, x0, y0 in zip(group, xs, ys):
                d.move_to([x0 + disp(x0, t), y0, 0])

        particles.add_updater(move_particles)

        # the piston that drives the wave
        piston = RoundedRectangle(
            corner_radius=0.06, width=0.2, height=Y_HI - Y_LO + 0.3,
            fill_color=anim.IRON, fill_opacity=0.55,
            stroke_color=anim.IRON, stroke_width=1.5,
        )
        piston.add_updater(lambda m: m.move_to(
            [PISTON_X + disp(PISTON_X, clock.get_value()), (Y_LO + Y_HI) / 2, 0]
        ))

        # the fixed measurement point
        probe = Line([PROBE_X, Y_LO - 0.15, 0], [PROBE_X, Y_HI + 0.15, 0],
                     color=anim.RED, stroke_width=4)
        probe_label = Text("microphone", font_size=26)
        probe_label.move_to([PROBE_X, 3.35, 0])

        # the oscilloscope
        axes = Axes(
            x_range=[0, T_END - T_TRACE, 2],
            y_range=[-1.4, 1.4, 1],
            x_length=11.0, y_length=2.3,
            axis_config={"color": anim.IRON, "include_ticks": False,
                         "stroke_width": 1.5},
            tips=False,
        ).move_to([0.1, -2.35, 0])
        t_label = Text("time", font_size=22)
        t_label.next_to(axes.x_axis.get_end(), DOWN, buff=0.2)
        p_label = Text("pressure", font_size=22).rotate(PI / 2)
        p_label.next_to(axes.y_axis, LEFT, buff=0.15)
        xt = MathTex("x(t)", color=anim.RED).scale(0.9)
        xt.move_to(axes.c2p(0.7, 1.05))

        trace = always_redraw(lambda: axes.plot(
            lambda u: pressure(u + T_TRACE),
            x_range=[0, max(clock.get_value() - T_TRACE, 1e-3), 0.01],
            color=anim.RED, stroke_width=2.6,
        ))
        pen = Dot(color=anim.RED, radius=0.055)
        pen.add_updater(lambda m: m.move_to(axes.c2p(
            max(clock.get_value() - T_TRACE, 0.0),
            pressure(max(clock.get_value(), T_TRACE)),
        )))

        # timeline: every play advances the clock at a linear rate, so world
        # time stays equal to clock time
        def tick(dt, *anims):
            self.play(
                clock.animate(rate_func=linear).set_value(clock.get_value() + dt),
                *anims,
                run_time=dt,
            )

        self.add(particles, piston)
        tick(1.0, FadeIn(particles), FadeIn(piston))
        tick(2.5)  # a beat to watch the pattern travel
        tick(1.0, FadeIn(probe), FadeIn(probe_label))
        self.add(trace, pen)
        tick(0.5, FadeIn(axes), FadeIn(t_label), FadeIn(p_label), FadeIn(xt))
        tick(T_END - clock.get_value())

anim.show(PressureWave)